In [2]:
!pip install datasets pyarrow pandas

In [3]:
from datasets import load_dataset

dataset = load_dataset(
    "ButterChicken98/plantvillage-image-text-pairs",
    split="train"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/365 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/344M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/20638 [00:00<?, ? examples/s]

In [4]:
df = dataset.to_pandas()
df.head()

,image,caption,captions
0,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato healthy,[A vibrant green and healthy tomato leaf with ...
1,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato Late blight,[A tomato leaf showing dark brown lesions and ...
2,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato healthy,[A vibrant green and healthy tomato leaf with ...
3,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato mosaic virus,[A tomato leaf with mosaic-like patterns of li...
4,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Pepper bell healthy,"[A fresh green bell pepper leaf with a smooth,..."


In [5]:
df.rename(columns={'caption': 'class_name'}, inplace=True)

In [6]:
print("Total classes:", df['class_name'].nunique())
print(df['class_name'].unique())

Total classes: 15
['Tomato healthy' 'Tomato Late blight' 'Tomato mosaic virus'
 'Pepper bell healthy' 'Potato Early blight' 'Tomato Early blight'
 'Tomato YellowLeaf Curl Virus' 'Tomato Target Spot'
 'Pepper bell Bacterial spot' 'Tomato Septoria leaf spot'
 'Tomato Spider mites Two spotted spider mite' 'Tomato Bacterial spot'
 'Potato Late blight' 'Tomato Leaf Mold' 'Potato healthy']


In [7]:
df['class_name'].value_counts()

,count
class_name,
Tomato YellowLeaf Curl Virus,3208
Tomato Bacterial spot,2127
Tomato Late blight,1909
Tomato Septoria leaf spot,1771
Tomato Spider mites Two spotted spider mite,1676
Tomato healthy,1591
Pepper bell healthy,1478
Tomato Target Spot,1404
Potato Late blight,1000


In [8]:
df['num_captions'] = df['captions'].apply(len)
df['num_captions'].value_counts()

,count
num_captions,
4,20638


In [9]:
all_captions = df['captions'].explode()

print("Total symptom descriptions:", len(all_captions))
print("Unique descriptions:", all_captions.nunique())

Total symptom descriptions: 82552
Unique descriptions: 60


Import libraries


In [12]:
!pip install transformers datasets scikit-learn pandas torch

In [14]:
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments
)

from datasets import Dataset

LOARD AND PREPARE DATASET


In [15]:
from datasets import load_dataset

dataset = load_dataset("ButterChicken98/plantvillage-image-text-pairs", split="train")

df = dataset.to_pandas()

df.head()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


,image,caption,captions
0,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato healthy,[A vibrant green and healthy tomato leaf with ...
1,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato Late blight,[A tomato leaf showing dark brown lesions and ...
2,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato healthy,[A vibrant green and healthy tomato leaf with ...
3,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato mosaic virus,[A tomato leaf with mosaic-like patterns of li...
4,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Pepper bell healthy,"[A fresh green bell pepper leaf with a smooth,..."


In [16]:
df = df[['caption']]

df.rename(columns={"caption":"text"}, inplace=True)

df.head()

,text
0,Tomato healthy
1,Tomato Late blight
2,Tomato healthy
3,Tomato mosaic virus
4,Pepper bell healthy


In [17]:
label_encoder = LabelEncoder()

df["label"] = label_encoder.fit_transform(df["text"])

df.head()

,text,label
0,Tomato healthy,13
1,Tomato Late blight,7
2,Tomato healthy,13
3,Tomato mosaic virus,14
4,Pepper bell healthy,1


In [18]:
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["text"],
    df["label"],
    test_size=0.2,
    random_state=42
)

In [20]:
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

train_encodings = tokenizer(
    list(train_texts),
    truncation=True,
    padding=True
)

test_encodings = tokenizer(
    list(test_texts),
    truncation=True,
    padding=True
)

In [21]:
train_dataset = Dataset.from_dict({
    "input_ids": train_encodings["input_ids"],
    "attention_mask": train_encodings["attention_mask"],
    "labels": list(train_labels)
})

test_dataset = Dataset.from_dict({
    "input_ids": test_encodings["input_ids"],
    "attention_mask": test_encodings["attention_mask"],
    "labels": list(test_labels)
})

DistilBERT tokenization

In [25]:
from transformers import DistilBertForSequenceClassification

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(label_encoder.classes_)
)

print("Model loaded successfully")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded successfully


In [26]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    logging_dir="./logs"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [27]:
from transformers import Trainer, TrainingArguments

In [28]:
trainer = Trainer(
    model =model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

print("Trainer created successfully")

Trainer created successfully


TRAINING AND EVALUVATE

In [30]:
trainer.train()

Step,Training Loss
500,0.065452
1000,0.005363
1500,0.002403
2000,0.001501
2500,0.001096
3000,0.000896


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3096, training_loss=0.012415019128693012, metrics={'train_runtime': 3699.5575, 'train_samples_per_second': 13.388, 'train_steps_per_second': 0.837, 'total_flos': 145229832073260.0, 'train_loss': 0.012415019128693012, 'epoch': 3.0})

In [35]:
from transformers.utils import logging
logging.disable_progress_bar()

In [36]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

In [38]:
results = trainer.predict(test_dataset)
print(results.metrics)

{'test_loss': 0.00047475245082750916, 'test_model_preparation_time': 0.0018, 'test_runtime': 84.0953, 'test_samples_per_second': 49.087, 'test_steps_per_second': 3.068}


MODEL ACURACY

In [39]:

from sklearn.metrics import accuracy_score

y_pred = results.predictions.argmax(-1)
y_true = results.label_ids

accuracy = accuracy_score(y_true, y_pred)

print("Model Accuracy:", accuracy)

Model Accuracy: 1.0


PREDICTING

In [40]:
text = "Tomato leaf bacterial spot"

inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

outputs = model(**inputs)

pred = torch.argmax(outputs.logits, dim=1)

print("Predicted Label:", pred.item())
print("Class Name:", label_encoder.inverse_transform([pred.item()])[0])

Predicted Label: 5
Class Name: Tomato Bacterial spot


SAVE MODEL

In [41]:
model.save_pretrained("plant_disease_model")
tokenizer.save_pretrained("plant_disease_model")

print("Model saved successfully")

Model saved successfully


In [42]:
from transformers import DistilBertForSequenceClassification, DistilBertTokenizerFast

model = DistilBertForSequenceClassification.from_pretrained("plant_disease_model")
tokenizer = DistilBertTokenizerFast.from_pretrained("plant_disease_model")

print("Model loaded successfully")

Model loaded successfully
